In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── GPU CHECK ────────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Go to Runtime → Change runtime type → T4 GPU")
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}  ({vram_gb:.1f} GB)")

# ── DRIVE MOUNT ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ANRA = Path("/content/drive/MyDrive/AnRa")
DRIVE_ANRA.mkdir(parents=True, exist_ok=True)
print(f"Drive: {DRIVE_ANRA}")

# ── FIND OR CLONE REPO ───────────────────────────────────────────────────────
# Search /content for the repo — works regardless of what git clone named it
REPO = None
for candidate in Path("/content").iterdir():
    if candidate.is_dir() and (candidate / "anra_paths.py").exists():
        REPO = candidate
        print(f"Repo found: {REPO}")
        break

if REPO is None:
    print("Cloning repo...")
    result = subprocess.run(
        ["git", "clone", "--quiet",
         "https://github.com/YOUR_USERNAME/An-Ra-the-new-AGI.git"],
        cwd="/content", capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{result.stderr}")
    # Find it again after clone
    for candidate in Path("/content").iterdir():
        if candidate.is_dir() and (candidate / "anra_paths.py").exists():
            REPO = candidate
            break
    if REPO is None:
        raise RuntimeError("Could not find repo after clone. Check GitHub URL above.")
    print(f"Repo cloned: {REPO}")
else:
    os.chdir(REPO)
    result = subprocess.run(
        ["git", "pull", "--quiet"], capture_output=True, text=True, cwd=str(REPO)
    )
    pulled = result.stdout.strip() or "already up to date"
    print(f"Git pull: {pulled}")

# ── SET PATH — fixes ALL ModuleNotFoundError for anra_paths ─────────────────
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)  # fixes subprocess !python calls too

# Verify anra_paths is importable
try:
    from anra_paths import inject_all_paths
    inject_all_paths()
    print(f"anra_paths: OK")
except Exception as e:
    raise RuntimeError(f"anra_paths still not importable: {e}\nREPO={REPO}")

# ── INSTALL DEPENDENCIES ─────────────────────────────────────────────────────
print("Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "tokenizers", "sympy", "networkx", "accelerate"],
    check=True
)
print("Dependencies: OK")

# ── STORE REPO PATH FOR ALL OTHER CELLS ─────────────────────────────────────
# Write REPO path to a temp file so other cells can read it without repeating discovery
Path("/tmp/anra_repo_path.txt").write_text(str(REPO))

print(f"\n{'='*50}")
print(f"SETUP COMPLETE")
print(f"Repo : {REPO}")
print(f"GPU  : {gpu_name}")
print(f"Drive: {DRIVE_ANRA}")
print(f"{'='*50}")
print("Run Cell 2 next.")


In [ ]:
import os, sys, shutil, torch
from pathlib import Path

# ── LOAD REPO PATH ───────────────────────────────────────────────────────────
_repo_file = Path("/tmp/anra_repo_path.txt")
if not _repo_file.exists():
    raise RuntimeError("Run Cell 1 first.")
REPO = Path(_repo_file.read_text().strip())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)

from anra_paths import inject_all_paths
inject_all_paths()

# ── PATHS ─────────────────────────────────────────────────────────────────────
DRIVE_ANRA   = Path("/content/drive/MyDrive/AnRa")
OUTPUT_V2    = REPO / "output" / "v2"
OUTPUT_V2.mkdir(parents=True, exist_ok=True)

LOCAL_BRAIN  = OUTPUT_V2 / "anra_v2_brain.pt"
LOCAL_TOK    = REPO / "tokenizer" / "tokenizer_v2.json"

DRIVE_BRAIN  = DRIVE_ANRA / "anra_v2_brain.pt"
DRIVE_TOK    = DRIVE_ANRA / "tokenizer_v2.json"

# ── RESTORE BRAIN ────────────────────────────────────────────────────────────
def restore(src, dst, label):
    if not src.exists():
        print(f"  {label}: not on Drive — will start fresh")
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_mtime >= src.stat().st_mtime:
        print(f"  {label}: already current locally")
        return True
    shutil.copy2(src, dst)
    print(f"  {label}: restored from Drive")
    return True

brain_found = restore(DRIVE_BRAIN, LOCAL_BRAIN, "Brain checkpoint")
tok_found   = restore(DRIVE_TOK,   LOCAL_TOK,   "Tokenizer")

# ── STATUS ───────────────────────────────────────────────────────────────────
print()
if brain_found and LOCAL_BRAIN.exists():
    try:
        ckpt = torch.load(LOCAL_BRAIN, map_location="cpu", weights_only=False)
        step      = ckpt.get("step", 0)
        best_loss = ckpt.get("best_loss", float("inf"))
        sessions  = ckpt.get("sessions_completed", 0)
        print(f"✓ Resuming from  step={step}  loss={best_loss:.4f}  sessions={sessions}")
    except Exception as e:
        print(f"✓ Checkpoint found but could not read metadata: {e}")
else:
    print("○ No checkpoint — training starts from scratch (step 0)")

if tok_found:
    size = LOCAL_TOK.stat().st_size // 1024
    print(f"✓ Tokenizer ready ({size}KB)")
else:
    print("○ Tokenizer not on Drive — using repo default")

print("\nRun Cell 3 next.")


In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── LOAD REPO PATH ───────────────────────────────────────────────────────────
_repo_file = Path("/tmp/anra_repo_path.txt")
if not _repo_file.exists():
    raise RuntimeError("Run Cell 1 first.")
REPO = Path(_repo_file.read_text().strip())
os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO)

# ── CONFIG ───────────────────────────────────────────────────────────────────
SESSION_MINUTES = 30   # change this if you want longer sessions

# ── TRAIN WITH LIVE OUTPUT ───────────────────────────────────────────────────
print(f"Starting {SESSION_MINUTES}-min training session...")
print("Metrics print every 10 steps.\n")
print("─" * 50)

# Use Popen so output streams live — you see every step printed immediately
process = subprocess.Popen(
    [
        sys.executable, "-m", "training.train_unified",
        "--mode", "session",
        "--session_minutes", str(SESSION_MINUTES),
    ],
    cwd=str(REPO),
    env={**os.environ, "PYTHONPATH": str(REPO), "PYTHONUNBUFFERED": "1"},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream every line as it comes
try:
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    process.terminate()
    print("\nTraining interrupted by user.")

process.wait()

print("─" * 50)
if process.returncode == 0:
    print("\n✓ Session complete. Run Cell 4 to save to Drive.")
else:
    print(f"\n✗ Training exited with code {process.returncode}")
    print("Most common fix: re-run Cell 1, then run Cell 3 again.")


In [ ]:
import os, sys, shutil, torch
from pathlib import Path

# ── LOAD REPO PATH ───────────────────────────────────────────────────────────
_repo_file = Path("/tmp/anra_repo_path.txt")
if not _repo_file.exists():
    raise RuntimeError("Run Cell 1 first.")
REPO = Path(_repo_file.read_text().strip())
os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO)

# ── PATHS ─────────────────────────────────────────────────────────────────────
DRIVE_ANRA  = Path("/content/drive/MyDrive/AnRa")
DRIVE_ANRA.mkdir(parents=True, exist_ok=True)
OUTPUT_V2   = REPO / "output" / "v2"

LOCAL_BRAIN = OUTPUT_V2 / "anra_v2_brain.pt"
LOCAL_TOK   = REPO / "tokenizer" / "tokenizer_v2.json"
LOCAL_EVAL  = OUTPUT_V2 / "anra_v2_eval_summary.json"

# ── SAVE — always overwrites same 3 files ────────────────────────────────────
def save_to_drive(src, label):
    if not src.exists():
        print(f"  {label}: not found locally, skipping")
        return
    dst = DRIVE_ANRA / src.name
    shutil.copy2(src, dst)
    size_kb = src.stat().st_size // 1024
    print(f"  ✓ {label} saved ({size_kb}KB)")

save_to_drive(LOCAL_BRAIN, "Brain checkpoint")
save_to_drive(LOCAL_TOK,   "Tokenizer")
save_to_drive(LOCAL_EVAL,  "Eval summary")

# ── PRINT WHAT'S ON DRIVE ────────────────────────────────────────────────────
print()
drive_brain = DRIVE_ANRA / "anra_v2_brain.pt"
if drive_brain.exists():
    try:
        ckpt      = torch.load(drive_brain, map_location="cpu", weights_only=False)
        step      = ckpt.get("step", 0)
        loss      = ckpt.get("best_loss", float("inf"))
        sessions  = ckpt.get("sessions_completed", 0)
        print(f"Drive checkpoint: step={step}  loss={loss:.4f}  sessions={sessions}")
    except Exception:
        print("Drive checkpoint: saved (could not read metadata)")

print("\n✓ Safe to close Colab.")
print("Next session: Runtime → Disconnect → open new session → run Cell 1 → Cell 2 → Cell 3 → Cell 4")


In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── LOAD REPO PATH ───────────────────────────────────────────────────────────
_repo_file = Path("/tmp/anra_repo_path.txt")
if not _repo_file.exists():
    raise RuntimeError("Run Cell 1 first.")
REPO = Path(_repo_file.read_text().strip())
os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO)

# ── MILESTONE SESSION ────────────────────────────────────────────────────────
# Only run this every ~5 sessions.
# Does: identity fine-tune → ouroboros → self-improvement → tests
# Takes ~90 minutes.

print("MILESTONE SESSION (~90 min)")
print("identity → ouroboros → self-improvement → tests\n")
print("─" * 50)

process = subprocess.Popen(
    [
        sys.executable, "-m", "training.train_unified",
        "--mode", "train",
        "--session_minutes", "90",
    ],
    cwd=str(REPO),
    env={**os.environ, "PYTHONPATH": str(REPO), "PYTHONUNBUFFERED": "1"},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    process.terminate()
    print("\nMilestone interrupted.")

process.wait()
print("─" * 50)

if process.returncode == 0:
    print("\n✓ Milestone complete. Run Cell 4 to save.")
else:
    print(f"\n✗ Exited with code {process.returncode}")
